In [23]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from glob import glob

In [5]:
data = pd.read_parquet("../data/raw/date=2025-12-23/beer-1766450668592.parquet")


# Combining Data

In [7]:
data

,plant_id,line_id,batch_id,ts,step,sensor,value,unit,in_spec
0,plantB,line2,batch-plantB-line2-1766447447,2025-12-22T23:51:12.899100Z,mashing,temp,65.830,C,True
1,plantB,line2,batch-plantB-line2-1766447447,2025-12-22T23:51:13.910902Z,packaging,count,147.000,units,False
2,plantB,line2,batch-plantB-line2-1766447447,2025-12-22T23:51:13.911046Z,packaging,fill_level,334.540,ml,True
3,plantB,line2,batch-plantB-line2-1766447447,2025-12-22T23:51:13.911087Z,packaging,cap_torque,16.020,Ncm,True
4,plantB,line2,batch-plantB-line2-1766447447,2025-12-22T23:51:13.911108Z,packaging,line_speed,138.700,units/min,True
5,plantB,line2,batch-plantB-line2-1766447447,2025-12-22T23:51:13.911127Z,packaging,reject_rate,1.670,%,True
6,plantA,line1,batch-plantA-line1-1766447449,2025-12-22T23:51:14.913975Z,mashing,temp,62.960,C,True
7,plantA,line1,batch-plantA-line1-1766447449,2025-12-22T23:51:15.922221Z,packaging,count,95.000,units,True
8,plantA,line1,batch-plantA-line1-1766447449,2025-12-22T23:51:15.922483Z,packaging,fill_level,340.700,ml,False
9,plantA,line1,batch-plantA-line1-1766447449,2025-12-22T23:51:15.922526Z,packaging,cap_torque,13.200,Ncm,True


In [11]:
files = glob("../data/raw/date=2025-12-23/*.parquet")

df = pd.concat((pd.read_parquet(f) for f in files), 
               ignore_index=True)

df.to_parquet("../data/combined/date=2025-12-23/beer_combined.parquet", index=False)

In [13]:
print(df.info())

df = df.drop_duplicates()


print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2840 entries, 0 to 2839
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   plant_id  2840 non-null   object 
 1   line_id   2840 non-null   object 
 2   batch_id  2840 non-null   object 
 3   ts        2840 non-null   object 
 4   step      2840 non-null   object 
 5   sensor    2840 non-null   object 
 6   value     2840 non-null   float64
 7   unit      2840 non-null   object 
 8   in_spec   2840 non-null   bool   
dtypes: bool(1), float64(1), object(7)
memory usage: 180.4+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2840 entries, 0 to 2839
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   plant_id  2840 non-null   object 
 1   line_id   2840 non-null   object 
 2   batch_id  2840 non-null   object 
 3   ts        2840 non-null   object 
 4   step      2840 non-null   object 
 5   sensor    284

In [14]:
com_data = pd.read_parquet("../data/combined/date=2025-12-23/beer_combined.parquet")

In [17]:
com_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2840 entries, 0 to 2839
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   plant_id  2840 non-null   object 
 1   line_id   2840 non-null   object 
 2   batch_id  2840 non-null   object 
 3   ts        2840 non-null   object 
 4   step      2840 non-null   object 
 5   sensor    2840 non-null   object 
 6   value     2840 non-null   float64
 7   unit      2840 non-null   object 
 8   in_spec   2840 non-null   bool   
dtypes: bool(1), float64(1), object(7)
memory usage: 180.4+ KB


In [20]:
com_data.head(5)

,plant_id,line_id,batch_id,ts,step,sensor,value,unit,in_spec
0,plantB,line2,batch-plantB-line2-1766509336,2025-12-23T17:34:36.550436Z,packaging,line_speed,137.10,units/min,True
1,plantB,line2,batch-plantB-line2-1766509336,2025-12-23T17:34:36.550559Z,packaging,reject_rate,2.01,%,True
2,plantA,line2,batch-plantA-line2-1766509337,2025-12-23T17:34:37.561694Z,fermentation,temp,22.00,C,True
3,plantA,line2,batch-plantA-line2-1766509337,2025-12-23T17:34:37.562231Z,fermentation,ph,3.96,pH,True
4,plantA,line2,batch-plantA-line2-1766509337,2025-12-23T17:34:37.562416Z,fermentation,gravity_degP,13.25,degP,True


### Dec

In [25]:
RAW_ROOT = Path("../data/raw")
COMBINED_ROOT = Path("../data/combined/december")
COMBINED_ROOT.mkdir(parents=True, exist_ok=True)

for date_dir in sorted(RAW_ROOT.glob("date=*")):
    date_str = date_dir.name.split("=", 1)[-1]
    try:
        dt = datetime.fromisoformat(date_str)
    except ValueError:
        continue

    if dt.month != 12:  
        continue

    parts = sorted(date_dir.glob("*.parquet"))
    if not parts:
        continue

    dfs = [pd.read_parquet(p) for p in parts]
    combined = pd.concat(dfs, ignore_index=True)

    out_path = COMBINED_ROOT / f"date={dt.date().isoformat()}.parquet"
    combined.to_parquet(out_path, index=False)
    print(f"Wrote {out_path} ({len(combined)} rows from {len(parts)} files)")


Wrote ../data/combined/december/date=2025-12-01.parquet (19140 rows from 957 files)
Wrote ../data/combined/december/date=2025-12-02.parquet (30560 rows from 1528 files)
Wrote ../data/combined/december/date=2025-12-03.parquet (20660 rows from 1033 files)
Wrote ../data/combined/december/date=2025-12-04.parquet (41240 rows from 2062 files)
Wrote ../data/combined/december/date=2025-12-05.parquet (6800 rows from 340 files)
Wrote ../data/combined/december/date=2025-12-11.parquet (7800 rows from 2 files)
Wrote ../data/combined/december/date=2025-12-15.parquet (8120 rows from 406 files)
Wrote ../data/combined/december/date=2025-12-16.parquet (31120 rows from 1556 files)
Wrote ../data/combined/december/date=2025-12-17.parquet (54160 rows from 2708 files)
Wrote ../data/combined/december/date=2025-12-18.parquet (40860 rows from 2043 files)
Wrote ../data/combined/december/date=2025-12-22.parquet (7520 rows from 376 files)
Wrote ../data/combined/december/date=2025-12-23.parquet (2840 rows from 142 